# Lecture 2: Language modeling basics and Rasp (code companion)

*Discrete Methods for Modern AI*

This notebook has the code examples from the lecture 2 slides, so you can run and change them yourself. It has three parts:

1. **Words as vectors**: word embeddings in spaCy.
2. **Self-attention from scratch**: embeddings, attention, causal masking, keys, queries and values, and transformer blocks in PyTorch.
3. **Thinking like a transformer**: a small implementation of Rasp, the reverse program from the lecture, and exercises.

**Setup.** Run this once in your environment (a terminal or a notebook cell):

```
pip install torch spacy numpy
python -m spacy download en_core_web_md
```

In [2]:
# Uncomment to install from inside the notebook:
!pip install torch spacy numpy
!python -m spacy download en_core_web_md

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-1.0.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached confection-1.3.4-py3-none-any.whl.metadata (19 kB)
  Using cached typer-0.28.0-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.5.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached cloudpathlib-0.26.0-py3-none-any.whl.metadata (16 kB)
  Using cached smart_open-9.0.2-py3-none-any.whl.metadata (25 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Usin

---
## Part 1: Words as vectors

Classical models treat words as discrete symbols. Modern deep learning models work in a **continuous space**: each word is a vector of real numbers, and similar words get nearby vectors.

In [3]:
import spacy

nlp = spacy.load("en_core_web_md")  ## has word vectors

coffee = nlp("coffee")[0]           ## one word (token)
print(coffee.vector.shape)          ## => (300,) real numbers
print(coffee.vector[:4])

(300,)
[-0.71252  -0.035167 -0.18155   0.38755 ]


In [4]:
tea, norway = nlp("tea")[0], nlp("Norway")[0]
print("coffee ~ tea:   ", round(coffee.similarity(tea), 2))     ## similar
print("coffee ~ Norway:", round(coffee.similarity(norway), 2))  ## unrelated

coffee ~ tea:    0.57
coffee ~ Norway: 0.06


### It's easy to make mistakes in continuous spaces

Tunisia and Norway are just nearby vectors. Look at the similarity between Tunisia and Morocco.

In [5]:
tunisia = nlp("Tunisia")[0]
print(tunisia.vector[:5])
print(norway.vector[:5])

print("Tunisia ~ Norway: ", round(tunisia.similarity(norway), 2))             ## close
print("Tunisia ~ coffee: ", round(tunisia.similarity(nlp("coffee")[0]), 2))   ## far
print("Tunisia ~ Morocco:", round(tunisia.similarity(nlp("Morocco")[0]), 2))  ## same vector!

[-0.60028 -0.24248  0.42605  0.58576  0.1129 ]
[-0.72509  0.2789   0.26907 -0.28551  0.32117]
Tunisia ~ Norway:  0.54
Tunisia ~ coffee:  0.02
Tunisia ~ Morocco: 1.0


> **Why 1.0?** `en_core_web_md` stores fewer vectors than words, so some words share a vector. Try a few other pairs of countries or cities. Which ones can the model not tell apart?

---
## Part 2: Self-attention from scratch

We follow the running example from the slides:

`<BOS> Norway is to the west of Sweden`

Each word gets an id, and an embedding matrix `E` maps ids to vectors. The embeddings here are **random and untrained**, so the numbers mean nothing yet. The point is the shapes and the computation.

In [6]:
import torch ## to install: pip install torch
torch.manual_seed(0)

### word embedding parameters and matrix E
E = torch.nn.Embedding(
    embedding_dim=768,   ##<--- dimensionality
    num_embeddings=3000, ##<--- # words
)

##e.g., Representation of our Input
X = E(torch.tensor([
    0, # <BOS>
    1, # Norway
    2, # is
    3, # to
    4, # the
    5, # west
    6, # of
    7, # Sweden
])) ### => matrix 8 * 768

x1 = X[1] ##-> initial representation of Norway
print(X.shape)
print(x1[:4])

torch.Size([8, 768])
tensor([0.4974, 0.2685, 1.4769, 0.3548], grad_fn=<SliceBackward0>)


### Self-attention

Every word scores every other word with a dot product. A softmax turns the scores into a probability distribution, and the new representation of each word is a weighted average of all the words.

In [7]:
### raw weights (dot product / matrix multiplication)
raw_Alpha = torch.matmul(X, X.transpose(0, 1))
### normalized via softmax, probability distribution
alpha = raw_Alpha.softmax(dim=-1)
### Final self attention representations
Y = torch.matmul(alpha, X)
### self attention representation of `Norway`
y1 = Y[1]

print(alpha.shape, Y.shape)
print("each row sums to 1:", alpha.sum(dim=-1))

torch.Size([8, 8]) torch.Size([8, 768])
each row sums to 1: tensor([1., 1., 1., 1., 1., 1., 1., 1.], grad_fn=<SumBackward1>)


> **Try it.** Print `alpha`. With random 768-dimensional vectors, each word attends almost only to itself. Why? (Hint: compare `X[1] @ X[1]` with `X[1] @ X[2]`.) Real models scale the scores by $1/\sqrt{d}$. Try `raw_Alpha / 768**0.5`.

### Causal attention: adding a mask

A language model that predicts the next word must not look at future words. We set the scores above the diagonal to $-\infty$ **before** the softmax, so they get weight 0.

In [8]:
### raw weights (dot product / matrix multiplication)
raw_Alpha = torch.matmul(X, X.transpose(0, 1))

### causal mask
mask = torch.triu_indices(8, 8, offset=1)
raw_Alpha[mask[0], mask[1]] = float('-inf')

### normalized via softmax, probability distribution
alpha = raw_Alpha.softmax(dim=-1)
### Final self attention representations
Y = torch.matmul(alpha, X)
### self attention representation of `Norway`
y1 = Y[1]

print(torch.isfinite(raw_Alpha).int())   ## lower triangular: word j only sees words 0..j
print(alpha[0])                     ## <BOS> can only attend to itself: weight 1

tensor([[1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1]], dtype=torch.int32)
tensor([1., 0., 0., 0., 0., 0., 0., 0.], grad_fn=<SelectBackward0>)


> **Check yourself.**
> - Which word's representation is the same with and without the mask?
> - What happens if you set the masked entries to $-\infty$ **after** the softmax instead? Try it.

### More parameters: keys, queries and values

Each word plays three roles, and each role gets its own linear map.

In [9]:
D = 768

### key, value, query parameters (linear layer)
W_k = torch.nn.Linear(D, D, bias=False)
W_q = torch.nn.Linear(D, D, bias=False)
W_v = torch.nn.Linear(D, D, bias=False)

key_rep   = W_k(X) #<-- rep. of X as keys
query_rep = W_q(X) #<-- rep. of X as queries
value_rep = W_v(X) #<-- rep. of X as values

### sample computation with parameters applied over `X`
alpha = torch.matmul(
    query_rep,
    key_rep.transpose(0, 1)
).softmax(dim=-1)

### same as before
Y = torch.matmul(alpha, value_rep)
print(Y.shape)

torch.Size([8, 768])


### Transformer blocks and a whole model

The slides show the `Block` class from [nanoGPT](https://github.com/karpathy/nanoGPT). To make it runnable here, we first define small versions of the pieces it uses: `CausalSelfAttention` (the attention above, with keys, queries, values, the mask, and several heads) and an `MLP`.

In [10]:
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 3000
    block_size: int = 64    ## longest input
    n_embd: int = 768
    n_head: int = 12
    n_layer: int = 4
    bias: bool = False

class LayerNorm(nn.LayerNorm):
    def __init__(self, ndim, bias):
        super().__init__(ndim, bias=bias)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)  ## q, k, v at once
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

    def forward(self, x):
        B, T, C = x.size()  ## batch, sequence length, embedding size
        q, k, v = self.c_attn(x).split(C, dim=2)
        ## split into heads: (B, n_head, T, C / n_head)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)  ## the causal mask
        y = y.transpose(1, 2).contiguous().view(B, T, C)               ## put the heads back together
        return self.c_proj(y)

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)

    def forward(self, x):
        return self.c_proj(F.gelu(self.c_fc(x)))  ## acts on each position separately

**A transformer block (nanoGPT).** Attention lets each position look at the others. The MLP then works on each position on its own. `x = x + ...` is the residual connection.

In [11]:
## nanoGPT: https://github.com/karpathy/nanoGPT
class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

**A transformer model is a stack of blocks.** This is a sketch, simplified from nanoGPT's `GPT` class.

In [12]:
## sketch, simplified from nanoGPT
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        V, d, L = config.vocab_size, config.n_embd, config.n_layer
        self.wte = nn.Embedding(V, d)                  ## word embeddings
        self.wpe = nn.Embedding(config.block_size, d)  ## positions
        self.blocks = nn.ModuleList(
            [Block(config) for _ in range(L)])         ## L layers
        self.ln_f = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, V, bias=False)

    def forward(self, idx):
        pos = torch.arange(idx.size(1))
        x = self.wte(idx) + self.wpe(pos)   ## input representations
        for block in self.blocks:           ## layer after layer
            x = block(x)
        return self.lm_head(self.ln_f(x))   ## scores for the next word

In [13]:
config = Config()
model = GPT(config)

idx = torch.tensor([[0, 1, 2, 3, 4, 5, 6, 7]])  ## <BOS> Norway is to the west of Sweden
logits = model(idx)
print(logits.shape)  ## (batch, positions, vocabulary): a score for every possible next word at every position
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

torch.Size([1, 8, 3000])
32,976,384 parameters


> **Go through this yourself.**
> - Where in `GPT` do positions come in, and what would go wrong without them?
> - Turn `logits[0, -1]` into a probability distribution over the next word after "Sweden".
> - Change `n_layer` and `n_embd` and watch the number of parameters.
> - The full nanoGPT code is at https://github.com/karpathy/nanoGPT. Compare it with the sketch above.

---
## Part 3: Thinking like a transformer (Rasp)

Rasp ([Weiss et al., 2021](https://arxiv.org/abs/2106.06981)) is a small programming language whose operations correspond to parts of a transformer:

| transformer | Rasp |
|---|---|
| attention: which positions to look at | `select` |
| attention: averaging what you look at | `aggr` |
| MLP: element-wise computation | ordinary element-wise operations (`+`, `-`, ...) |

Here is the simplified version of a block from the lecture:

```python
class SimpleBlock(nn.Module):
    def forward(self, x):
        x = x + self.attn(x) #<--- looking around
        x = x + self.mlp(x)  #<--- element-wise
        #   | residual
        return x
```

### A small implementation of Rasp

Credits: https://github.com/apple/ml-np-rasp

In [14]:
## credits: https://github.com/apple/ml-np-rasp
import numpy as np

def indices(x):
    return np.arange(len(x), dtype=int)

def select(k, q, pred):
    s = len(k)
    A = np.zeros((s, s), dtype=bool)
    for i in range(s):
        for j in range(s): A[i, j] = pred(q[i], k[j])
    return A

def sel_width(A):
    return np.dot(A, np.ones(len(A))).astype(int)

def aggr(A, v):
    out = np.dot(A, v)
    norm = sel_width(A)
    out = np.divide(out, norm, out=np.zeros(len(out)), where=(norm != 0))
    return out.astype(int)

`select` builds an attention pattern: a boolean matrix where `A[i, j]` says whether position `i` attends to position `j`. `aggr` averages the values at the selected positions. `sel_width` counts them.

In [15]:
tokens = np.array([1, 2, 3, 0])

A = select(tokens, tokens, lambda q, k: k == 0)  ## every position attends to the 0
print(A.astype(int))
print("sel_width:", sel_width(A))
print("aggr of indices:", aggr(A, indices(tokens)))

[[0 0 0 1]
 [0 0 0 1]
 [0 0 0 1]
 [0 0 0 1]]
sel_width: [1 1 1 1]
aggr of indices: [3 3 3 3]


### Reversing a string

`0` is the end symbol (EOS). The program has two layers:

1. **Layer 1.** Every position attends to the end symbol to find the **length**. An MLP then computes `length - pos`, which is the position to copy from.
2. **Layer 2.** Every position attends to its target position and copies that token.

In [16]:
tokens = np.array([1, 2, 3, 0])

pos = indices(tokens)
EOS = 0

#### layer 1
# attn layer
length = aggr(
    select(tokens, tokens, lambda q, k: k == EOS),
    pos
)
targets = length - pos #<--- MLP

#### layer 2
output = aggr(
    select(pos, targets, lambda k, v: k == v),
    tokens
)

print("pos:    ", pos)
print("length: ", length)
print("targets:", targets)
print("output: ", output)   ## => [0 3 2 1]

pos:     [0 1 2 3]
length:  [3 3 3 3]
targets: [3 2 1 0]
output:  [0 3 2 1]


> **Check yourself.** Print the two attention patterns (the two `select` matrices). Which position does each row attend to? Try a longer input such as `[5, 4, 7, 1, 0]`.

---
## Exercises: programs to write in Rasp

Use only `indices`, `select`, `aggr`, `sel_width` and element-wise operations (no Python loops over positions, and no `len`, `sorted`, etc.). Afterwards, ask: **how many layers and heads does your program need?**

| | task | example |
|---|---|---|
| 1 | **length**, without an end symbol | `[3,1,4,1,5]` → `[5,5,5,5,5]` |
| 2 | **histogram**: count of each token | `[3,1,4,1,5]` → `[1,2,1,2,1]` |
| 3 | **shift** right by one | `[3,1,4,1,5]` → `[0,3,1,4,1]` |
| 4 | **reverse**, without an end symbol | `[3,1,4,1,5]` → `[5,1,4,1,3]` |
| 5 | **sort** distinct tokens | `[3,1,4,5,2]` → `[1,2,3,4,5]` |
| 6 | **sort** with repeats *(harder)* | `[3,1,4,1,5]` → `[1,1,3,4,5]` |

Fill in each function and run the check cell at the end.

In [17]:
def length(tokens):
    ## your code here
    return None

def histogram(tokens):
    ## your code here
    return None

def shift(tokens):
    ## your code here
    return None

def reverse(tokens):
    ## your code here (hint: you can reuse `length`)
    return None

def sort_distinct(tokens):
    ## your code here
    return None

def sort(tokens):
    ## your code here (harder)
    return None

In [18]:
tests = [
    (length,        [3, 1, 4, 1, 5], [5, 5, 5, 5, 5]),
    (histogram,     [3, 1, 4, 1, 5], [1, 2, 1, 2, 1]),
    (shift,         [3, 1, 4, 1, 5], [0, 3, 1, 4, 1]),
    (reverse,       [3, 1, 4, 1, 5], [5, 1, 4, 1, 3]),
    (sort_distinct, [3, 1, 4, 5, 2], [1, 2, 3, 4, 5]),
    (sort,          [3, 1, 4, 1, 5], [1, 1, 3, 4, 5]),
]

for f, x, expected in tests:
    out = f(np.array(x))
    if out is None:
        status = "not done yet"
    elif list(np.asarray(out)) == expected:
        status = "correct"
    else:
        status = f"wrong: got {list(np.asarray(out))}, expected {expected}"
    print(f"{f.__name__:14} {status}")

length         not done yet
histogram      not done yet
shift          not done yet
reverse        not done yet
sort_distinct  not done yet
sort           not done yet
